# Variable-Channel Spatial ASR — 재현 노트북마이크 1~4개를 **하나의 가중치**로 처리해 전사, 익명 화자 ID, 방위각·거리 분포를함께 내는 벤치마크입니다.이 노트북은 두 가지를 보여줍니다.1. **저장소에 커밋된 베이스라인 숫자가 재현된다** — GPU 없이, 표준 라이브러리만으로2. **프런티어 ASR을 External Reference로 붙이는 방법** — GPU 필요1~5번 셀은 무료 CPU 런타임에서 돌아갑니다. 6번만 GPU가 필요합니다.저장소: https://github.com/hd1534/variable-channel-spatial-asr

## 1. 저장소와 데이터 받기데이터셋이 저장소 안에 있어서 clone 한 번이면 끝납니다 (약 372 MB).

In [ ]:
!git clone --depth 1 https://github.com/hd1534/variable-channel-spatial-asr.git%cd variable-channel-spatial-asr!ls -la data/eval-dev

## 2. 무결성 검증`manifest.json`이 파일마다 SHA256을 들고 있습니다. 받은 게 생성된 것과 같은지직접 대조합니다. 하나라도 어긋나면 아래 숫자를 믿으면 안 됩니다.

In [ ]:
import hashlib, json, osm = json.load(open("data/eval-dev/manifest.json", encoding="utf-8"))files = m["files"]items = list(files.items()) if isinstance(files, dict) else [(f["path"], f) for f in files]bad = missing = checked = 0total = 0for path, rec in items:    want = rec if isinstance(rec, str) else (rec.get("sha256") or rec.get("hash"))    full = os.path.join("data/eval-dev", path)    if not os.path.isfile(full):        # RIR 번들은 의도적으로 커밋하지 않는다 (채점이 읽지 않는 생성 내부물)        if "/rir/" in path or path.startswith("rir/"):            continue        missing += 1        continue    h = hashlib.sha256()    with open(full, "rb") as fh:        for chunk in iter(lambda: fh.read(1 << 20), b""):            h.update(chunk)    total += os.path.getsize(full)    checked += 1    if want and h.hexdigest() != want:        bad += 1        print("  해시 불일치:", path)print(f"검증 {checked}개 / {total/1024/1024:.1f} MB")print(f"불일치 {bad}개, 누락 {missing}개")print()print("official        :", m.get("official"))print("차단 사유       :", m.get("official_blockers") or "없음")print("시드            :", m.get("seed"))print("RIR 엔진        :", m.get("rir_engine"))print("음원            :", m.get("source_mode"))assert bad == 0, "해시가 어긋났다. 다시 clone 하라."

## 3. 채점기 검증 — GPU도 의존성도 없이`evaluate.py`는 **표준 라이브러리만으로** 동작합니다. numpy는 선택적 가속일 뿐이고,numpy를 막아도 테스트가 전부 통과합니다. 채점 결과를 재현하는 데 아무 설치도필요 없다는 뜻입니다.

In [ ]:
!python3 -m unittest discover -s tests 2>&1 | tail -4

In [ ]:
# numpy 를 실제로 막고 다시 돌린다 -- 순수 파이썬 경로가 살아 있는지 확인import importlib.abc, io, subprocess, sys, textwrapprobe = textwrap.dedent('''    import importlib.abc, io, sys, unittest    class Block(importlib.abc.MetaPathFinder):        def find_spec(self, name, path=None, target=None):            if name == "numpy" or name.startswith("numpy."):                raise ImportError("numpy blocked")            return None    sys.meta_path.insert(0, Block())    sys.path[:0] = [".", "tests"]    import evaluate, test_evaluate    print("HAS_NUMPY =", evaluate.HAS_NUMPY)    r = unittest.TextTestRunner(stream=io.StringIO()).run(        unittest.TestLoader().loadTestsFromModule(test_evaluate))    print(f"{r.testsRun} tests, {len(r.failures)} failures, {len(r.errors)} errors")''')print(subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True).stdout)

## 4. 커밋된 베이스라인 숫자 재현`results/baseline2.json`에 들어 있는 숫자가 이 데이터에서 실제로 나오는지 확인합니다.- `LIMIT = 48` — 몇 분. 파이프라인이 도는지만 봅니다 (숫자는 부분집합이라 커밋값과 다름)- `LIMIT = None` — 864케이스 전부. CPU로 10분 안팎. **커밋된 숫자와 정확히 일치해야 합니다**

In [ ]:
LIMIT = 48   # 전체 재현은 None 으로 바꾼다

In [ ]:
!pip -q install "transformers==4.46.3" "soundfile==0.13.1" 2>&1 | tail -2

In [ ]:
import subprocess, syscmd = [sys.executable, "-m", "baselines.dsp_srp_phat",       "--eval-dir", "data/eval-dev", "--out", "runs/b2.jsonl",       "--manifest", "data/eval-dev/meta.jsonl", "--scope", "causal", "--track", "1"]if LIMIT:    cmd += ["--limit", str(LIMIT)]import os; os.makedirs("runs", exist_ok=True)print(subprocess.run(cmd, capture_output=True, text=True).stdout[-600:])

`--limit` 를 쓸 때는 **정답도 같이 잘라야 합니다.** 안 그러면 예측하지 않은 케이스가전부 결측으로 처리되어 NLL 이 페널티 바닥(27.63)까지 치솟습니다. 평가기는 정답에 있는모든 케이스를 채점하도록 설계돼 있고, 그건 "답하기 싫은 시점을 비워두는" 전략을막기 위한 규칙입니다.

In [ ]:
import jsonGT = "data/eval-dev/ground_truth.jsonl"if LIMIT:    have = {json.loads(l)["case_id"] for l in open("runs/b2.jsonl", encoding="utf-8")}    rows = [l for l in open(GT, encoding="utf-8") if json.loads(l)["case_id"] in have]    GT = "runs/gt_subset.jsonl"    open(GT, "w", encoding="utf-8").writelines(rows)    print(f"정답을 {len(rows)}케이스로 축소 (예측과 일치시킴)")else:    print("전체 864케이스로 채점한다")

In [ ]:
!python3 evaluate.py --pred runs/b2.jsonl --gt $GT --track 1 --out runs/b2.scores.json > /dev/nullimport jsongot = json.load(open("runs/b2.scores.json"))["metrics"]ref = json.load(open("results/baseline2.json"))["metrics"]print(f"{'지표':28s}{'재현':>12s}{'커밋됨':>12s}")print("-" * 52)for k in ("wer", "azimuth_mae_deg", "azimuth_acc_10deg", "distance_mae_m",          "pos_error_2d_m", "azimuth_nll", "distance_nll",          "azimuth_interval_width_deg", "rtf"):    f = lambda v: "N/A" if v is None else f"{v:.4f}"    print(f"{k:28s}{f(got.get(k)):>12s}{f(ref.get(k)):>12s}")if LIMIT is None:    same = all(got[k] == ref[k] for k in ref if ref[k] is not None)    print()    print("커밋된 숫자와 완전히 동일:", same)else:    print()    print(f"LIMIT={LIMIT} 부분집합이다. 분포 지표(NLL, 구간폭)는 전체값에 가깝게 나오고")    print("WER 과 MAE 는 발화와 방위 분포가 치우쳐서 다르다. 전체 비교는 LIMIT=None.")

## 5. 이 벤치마크의 핵심 그림 — 마이크를 늘리면 무슨 일이 일어나는가방위각 정확도를 채널 수별로 봅니다. **M=2에서 M=3으로 가는 구간**이 핵심입니다.마이크 2개는 두 마이크를 잇는 축 양쪽의 시간차가 같아서 앞뒤를 구분하지 못합니다.정답이 양봉 분포라 점 추정 오차가 큽니다. 비공선 세 번째 마이크가 그 모호성을 깹니다.

In [ ]:
import json, math, sys, collections, statistics as stsys.path.insert(0, ".")import spectruth = {}for line in open("data/eval-dev/ground_truth.jsonl", encoding="utf-8"):    r = json.loads(line)    truth[r["case_id"]] = {f["eval_time_sec"]: f["azimuth_deg"] for f in r.get("frames", [])}err = collections.defaultdict(list); nll = collections.defaultdict(list)for line in open("runs/b2.jsonl", encoding="utf-8"):    r = json.loads(line)    if r.get("record_type") != spec.RECORD_TYPE_TIMEPOINT:        continue    t = truth.get(r["case_id"], {}).get(r["eval_time_sec"])    if t is None:        continue    M = spec.parse_case_id(r["case_id"])[1].count("1")    err[M].append(spec.circular_error_deg(t, r["pred_azimuth_deg"]))    p = r.get("azimuth_probs")    if p:        nll[M].append(-math.log(max(p[spec.azimuth_bin_index(t)], spec.NLL_EPSILON)))print(f"{'마이크 수':>10s}{'Azimuth MAE':>14s}{'NLL':>9s}{'유효 bin':>10s}")print("-" * 44)for M in sorted(err):    a = st.mean(nll[M]) if nll[M] else float("nan")    print(f"{M:>10d}{st.mean(err[M]):13.2f}°{a:9.4f}{math.exp(a):10.2f}")print()print(f"균일 분포 기준선: NLL = ln 36 = {math.log(36):.4f}, 유효 bin 36, 우연 MAE 90°")

## 6. External Reference — 프런티어 ASR (GPU 필요)**런타임 → 런타임 유형 변경 → GPU** 로 바꾸고 실행하세요.`Qwen3-ASR-1.7B`는 통제 베이스라인과 **별도 표**에 들어갑니다. Controlled Baseline1·2는 같은 동결 ASR을 쓰고 앞단만 달라서 차이를 앞단 탓으로 돌릴 수 있지만,ASR 자체가 다른 모델을 같은 표에 넣으면 그 비교가 깨집니다.나올 결과를 미리 말하면 — **전사는 우리보다 잘하고, 공간 지표는 전부 N/A**입니다.단일 채널 입력이라 마이크를 더 줘도 달라지지 않습니다. 이 과제의 핵심 축을프런티어 모델이 다루지 못한다는 것 자체가 보고할 결과입니다.> 아래 셀은 **검증되지 않았습니다.** 모델 카드에서 정확한 로딩 방식과 라이선스를> 확인하고 고치세요: https://huggingface.co/Qwen/Qwen3-ASR-1.7B

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 런타임이 아니다"

In [ ]:
# [미검증] 모델 카드의 로딩 방식을 확인한 뒤 고쳐서 쓸 것MODEL = "Qwen/Qwen3-ASR-1.7B"import soundfile as sf, torch, globfrom transformers import AutoProcessor, AutoModelForSpeechSeq2Seqproc = AutoProcessor.from_pretrained(MODEL)model = AutoModelForSpeechSeq2Seq.from_pretrained(    MODEL, torch_dtype=torch.float16, device_map="auto")path = sorted(glob.glob("data/eval-dev/audio/*.wav"))[0]audio, sr = sf.read(path, dtype="float32", always_2d=True)mono = audio[:, 0]          # 단일 채널 -- 이 모델은 배열을 쓰지 않는다inputs = proc(mono, sampling_rate=sr, return_tensors="pt").to(model.device)with torch.no_grad():    ids = model.generate(**inputs, max_new_tokens=256)print(proc.batch_decode(ids, skip_special_tokens=True)[0])

## 7. 다음 단계 — 제안 모델 학습베이스라인이 서 있는 자리를 보면 어디를 노려야 하는지 분명합니다.| 목표 | 베이스라인 (B2) | 이론적 바닥 | 난이도 ||---|---:|---:|---|| **거리 NLL** | **2.0794** | 0 | 정확히 `ln 8`. 아무것도 안 하는 값 || **거리 구간 폭** | **4.5 m** | 0.2 m | support 전체를 깔고 있음 || **방위 구간 폭** | 246.5° | 10° | 6.5°로 맞히면서 190°를 깐다 || 방위 NLL | 2.3173 | 0 | || WER | 0.2541 | — | || 방위 MAE (M≥3) | 6.49° | — | 고전 DSP가 이미 천장. **건드리지 말 것** |거리는 베이스라인이 **시도조차 하지 않았습니다** (`distance_n_train: 0`, 균일 분포 출력).세션 음량이 무작위화돼 레벨 단서는 없지만, **DRR은 신호 내부의 비율이라 살아남습니다.**데이터에 같은 방에서 거리만 다른 대조(`has_same_room_distance_contrast`)가 들어 있어학습 문제가 잘 정의돼 있습니다.제안 모델은 `TAC` + `GRU` 구조입니다. 기획안 7.3의 `Recipe 1`을 보세요.